# Chapter 01: Working with text data
## 1.1 Download datasets for this chapter

Run the following code to download the dataset *the verdict* by Edith Wharton.

- After the first run, you should see *the-verdict.txt* file locally.
- If you couldn't download from the [url](https://raw.gitcode.com/GitHub_Trending/ll/LLMs-from-scratch/raw/main/ch02/01_main-chapter-code/the-verdict.txt) (for some reason about network), try to copy the raw text into a new file called *the-verdict.txt*.
- To check if the download is successful, use `len` to get the length of the text, which should be `20479`.

In [1]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "utils" / "paths.py").exists():
        sys.path.append(str(_p)); break
else:
    raise RuntimeError("repository root not found; start Jupyter inside the repo")
from utils.paths import data_path, repo_path

import os
import urllib.request

if not os.path.exists(data_path("the-verdict.txt")):
    url = ("https://raw.gitcode.com/GitHub_Trending/ll/LLMs-from-scratch/raw/main/ch02/01_main-chapter-code/the-verdict.txt")
    file_path = data_path("the-verdict.txt")
    urllib.request.urlretrieve(url, file_path)

In [2]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "utils" / "paths.py").exists():
        sys.path.append(str(_p)); break
else:
    raise RuntimeError("repository root not found; start Jupyter inside the repo")
from utils.paths import data_path, repo_path
with open(data_path("the-verdict.txt"), "r") as f:
    raw_text = f.read()
print(len(raw_text))

20479


## 1.2 Tokenize the text
You can use `re` to split the example text, in order to figure out how tokenizer works.
- Punctuation should be considered as separated tokens.
- Spaces should not be considered as tokens.

After the example test, you can use `re` to tokenize the dataset.

In [3]:
import re

text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [4]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])
print(len(preprocessed))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
4690


## 1.3 Convert tokens into token IDs

**Vocabulary** is a double map from tokens to number IDs. Each token should have, and must have, only one ID number. By doing so, we can use numbers to store the dataset text.
- Use `set` to clean duplicate tokens.
- Create `vocab` to convert tokens into token IDs.


In [5]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

vocab = {token:integer for integer,token in enumerate(all_words)}
print(vocab)

1130
{'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29, 'Devonshire': 30, 'Don': 31, 'Dubarry': 32, 'Emperors': 33, 'Florence': 34, 'For': 35, 'Gallery': 36, 'Gideon': 37, 'Gisburn': 38, 'Gisburns': 39, 'Grafton': 40, 'Greek': 41, 'Grindle': 42, 'Grindles': 43, 'HAD': 44, 'Had': 45, 'Hang': 46, 'Has': 47, 'He': 48, 'Her': 49, 'Hermia': 50, 'His': 51, 'How': 52, 'I': 53, 'If': 54, 'In': 55, 'It': 56, 'Jack': 57, 'Jove': 58, 'Just': 59, 'Lord': 60, 'Made': 61, 'Miss': 62, 'Money': 63, 'Monte': 64, 'Moon-dancers': 65, 'Mr': 66, 'Mrs': 67, 'My': 68, 'Never': 69, 'No': 70, 'Now': 71, 'Nutley': 72, 'Of': 73, 'Oh': 74, 'On': 75, 'Once': 76, 'Only': 77, 'Or': 78, 'Perhaps': 79, 'Poor': 80, 'Professional': 81, 'Renaissance': 82

Now, create a tokenizer class as below.
- The `encode` turns text into token IDs.
- The `decode` turns token IDs into text.

You can create an instance to test the tokenizer class.

In [6]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()} # reverse mapping
    
    # tokens -> ids
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    # ids -> tokens
    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [7]:
tokenizer = SimpleTokenizerV1(vocab)
test_text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(test_text)
print(ids)
print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


## 1.4 Add special context tokens
Using special context tokens may solve some problems:
- Processing new tokens that the dataset's vocabulary doesn't carry.
- Let LLM know when to start, and when to end.

We can use the following special tokens to deal with these problems:
- `[BOS]` marks the beginning of the text.
- `[EOS]` marks the end of the text (to concatenate multiple unrelated texts).
- `[PAD]` doesn't carry any information, it just **pads the text to the batch size**.
- `[UNK]` represents tokens that are not included in the vocabulary.

The following tokenizer class added `<|endoftext|>` (as `[EOS]`) and `<|unk|>`.

In [8]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()} # reverse mapping
    
    # tokens -> ids
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else "<|unk|>"
                        for item in preprocessed] 
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    # ids -> tokens
    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [9]:
tokenizer = SimpleTokenizerV2(vocab)
text1 = "Hello, do you like tea?" 
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

ids = tokenizer.encode(text)
print(ids)
# the token "Hello" and "palace" are marked as "<|unk|>"
print(tokenizer.decode(ids))

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## 1.5 Byte pair encoding

The popular LLMs such as the GPT series use **Byte Pair Encoding** (BPE) algorithms to tokenize a text. We will use **Tiktoken** in the following implementation (please install it first using `pip`). You can also try it online at [tiktokenizer.com](https://tiktokenizer.com/) to see how BPE works.

We can illustrate the BPE algorithms as follows
1. Suppose the initial vocabulary $V$ contains all basic characters
$$V=\{c_1,c_2,\cdots,c_n\}$$
2. For every adjacent character pair $(x,y)$, compute its *frequency* $f(x,y)$ in the present sequence $S=\{s_1,s_2,\cdots,s_m\}$
$$f(x,y)=count[(s_i,s_{i+1})=(x,y)]$$
3. During the $i$th iteration, update the vocabulary as
$$V\leftarrow V\cup\{x^*,y^*\},\quad (x^*,y^*)=\arg\max_{(x,y)} f(x,y)$$
4. Stop when any of the following conditions is met.
    - The vocabulary is big enough that $|V|=k$
    - The maximum frequency $\max f(x,y)$ is lower than the limit $t$

Therefore, BPE adds character pairs to the vocabulary to represent more unusual words. The BPE algorithm requires training, so we will use `gpt2` weights from OpenAI below.

In [10]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.")

# turn the text into token IDs by BPE 
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

# turn the token IDs back
strings = tokenizer.decode(integers)
print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


## 1.6 Data sampling with a sliding window
Since LLMs require the **context** to predict, one possible way is to set a sliding window for prediction. For example, we want to use the previous $k$ tokens to predict the next token, where $k$ is called *sliding window* and these $k$ tokens are called *context*.

Specifically, this can be seen as a **supervised learning** task, where
- The $(i)\sim (i+k)$ th tokens are inputs $x_i$.
- The $(i+1)\sim (i+k+1)$ th tokens are labels $y_i$ for $x_i$.
- The goal is to use batches such as $(x_i,y_i)$ to train the model.

This data sampling process ($k=4$) can be illustrated as follows.


In [11]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "utils" / "paths.py").exists():
        sys.path.append(str(_p)); break
else:
    raise RuntimeError("repository root not found; start Jupyter inside the repo")
from utils.paths import data_path, repo_path
with open(data_path("the-verdict.txt"), "r", encoding="utf-8") as f:
    raw_text = f.read()

# using BPE tokenizer to encode
enc_text = tokenizer.encode(raw_text)

# sliding window k=4
context_size = 4

x = enc_text[:context_size]
y = enc_text[1:context_size+1]

# slide for 5 times as example
for i in range(5):
    print(f"x{i}:{x}")
    print(f"y{i}:{y}")
    x = y
    y = enc_text[(i+2):context_size+i+2]

x0:[40, 367, 2885, 1464]
y0:[367, 2885, 1464, 1807]
x1:[367, 2885, 1464, 1807]
y1:[2885, 1464, 1807, 3619]
x2:[2885, 1464, 1807, 3619]
y2:[1464, 1807, 3619, 402]
x3:[1464, 1807, 3619, 402]
y3:[1807, 3619, 402, 271]
x4:[1807, 3619, 402, 271]
y4:[3619, 402, 271, 10899]


Another approach uses all previous tokens as the context to predict the next token.

To make the data processing more efficient, we could use `tensors` from `torch`.

In [14]:
import torch

# run the following code to make sure you already installed PyTorch
# print(torch.__version__)

from torch.utils.data import Dataset, DataLoader

# Dataset is a basic data class in PyTorch
# to create a new class from Dataset, yours must contain "__init__()", "__len__()" and "__getitem__()" function
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

The `GPTDatasetV1` class requires a dataloader to add data from the dataset.

In [13]:
# create a dataloader with batch_size, max_length, stride etc.
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

Now we could test with `batch_size=1`, `max_length=4` and `stride=1`.

In [15]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "utils" / "paths.py").exists():
        sys.path.append(str(_p)); break
else:
    raise RuntimeError("repository root not found; start Jupyter inside the repo")
from utils.paths import data_path, repo_path
with open(data_path("the-verdict.txt"), "r", encoding="utf-8") as f:
    raw_text = f.read()

# here, max_length refers to context window
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


Generally, we put batches together as a two-dimensional tensor input and target. To prevent the **over-fitting** problem, we usually set `stride=max_length` to make sure there's no window overlap.

In [16]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 1.7 Token embeddings

We've turned tokens into token IDs, but machines prefer real vectors to do calculations. Therefore, we need **embedding layers** to turn these integer IDs into $d$-dimensional vectors.

There are many ways to do embedding, and we choose `torch.nn.Embedding` for simplicity. 

In [17]:
vocab_size = 6
output_dim = 3

torch.manual_seed(325)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 1.1754, -0.4631, -1.0172],
        [ 1.8708, -1.3666,  0.2764],
        [-0.4429,  0.3375, -0.6269],
        [-2.3159,  1.5473,  0.9046],
        [ 0.3938, -0.3031, -0.1332],
        [-0.3103, -1.3748, -0.6151]], requires_grad=True)


Usually, the embedding layer includes a matrix with a shape of `vocab_size*output_dim`. The $i$ th row of the weight matrix corresponds to token ID $i+1$, as the following example shows.

In [18]:
input_ids = torch.tensor([3, 2, 5])
print(embedding_layer(input_ids))

tensor([[-2.3159,  1.5473,  0.9046],
        [-0.4429,  0.3375, -0.6269],
        [-0.3103, -1.3748, -0.6151]], grad_fn=<EmbeddingBackward0>)


## 1.8 Encoding word positions

The embedding layer converts IDs into identical vector representations regardless of where they are located in the input sequence. We want to add positional information during the embedding process to identify the same words appearing at different positions in the text.

The popular ways include **Triangle positional embedding** or **RoPE**, here we use the absolute position embedding just as what GPT-2 did.
- Remember carefully: The token embedding layer shares the same `out_dim` as the position embedding layer.

In [19]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

#---------------------------------------------------------------------
# to better understand the shape of tensors, see how outputs look like
#---------------------------------------------------------------------
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length,stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

# input tensor with batch_size=8, max_length=4
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

# input tensor after token embedding with batch_size=8, max_length=4
token_embeddings = token_embedding_layer(inputs)
print("\nInputs shape after token embedding:\n",token_embeddings.shape)

# position embedding
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print("\nPosition embedding layer's shape:\n",pos_embeddings.shape)

# ADD token embedding tensor with position embedding tensor
input_embeddings = token_embeddings + pos_embeddings
print("\nFinal input shape:\n",input_embeddings.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])

Inputs shape after token embedding:
 torch.Size([8, 4, 256])

Position embedding layer's shape:
 torch.Size([4, 256])

Final input shape:
 torch.Size([8, 4, 256])
